# LSTM Hyperparameter Tuning Notebook

This notebook is a minimal LSTM-only version of the uploaded notebook. It keeps the leakage-safe per-ticker feature scaling and target normalization, then tests 36 LSTM hyperparameter combinations and reports the top 3 combinations for each key metric.

**Run order:** run all cells from top to bottom. Make sure the CSV is in the same folder as this notebook.


In [1]:
# Optional: only run this if torch is not installed.
# In Jupyter, shell commands need a leading "!".
# !pip install --upgrade torch


In [2]:
import os
import itertools
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD & SORT
# ─────────────────────────────────────────────────────────────────────────────
# This will use the first file that exists, so it works with either filename.
CSV_CANDIDATES = [
    "feature_selected_dataset.csv",
    "feature_selected_dataset (1).csv",
]

csv_path = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)

if csv_path is None:
    print("Current folder contents:")
    print(os.listdir())
    raise FileNotFoundError(
        "Could not find feature_selected_dataset.csv or feature_selected_dataset (1).csv. "
        "Put the CSV in the same folder as this notebook, or edit CSV_CANDIDATES."
    )

print(f"Loading data from: {csv_path}")

df = pd.read_csv(csv_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

TARGET_COL = "target_next_return"

id_cols = ["ticker", "date"]
target_cols = [c for c in ["target_next_return", "target_next_price", "target_5d_return"] if c in df.columns]
feature_cols = [c for c in df.columns if c not in id_cols + target_cols]

print(f"Shape      : {df.shape}")
print(f"Target     : {TARGET_COL}")
print(f"# Features : {len(feature_cols)}")
print(f"Tickers    : {df['ticker'].nunique()}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. METRICS
# ─────────────────────────────────────────────────────────────────────────────
def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()

def sharpe_ratio_from_predictions(y_true, y_pred, annualization=252):
    positions = np.sign(y_pred)
    strategy_returns = positions * y_true
    std = np.std(strategy_returns)
    return np.nan if std == 0 else (np.mean(strategy_returns) / std) * np.sqrt(annualization)

def regression_metrics(y_true, y_pred, include_sharpe=True):
    mse = mean_squared_error(y_true, y_pred)
    metrics = {
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Directional_Accuracy": directional_accuracy(y_true, y_pred),
    }
    if include_sharpe:
        metrics["Sharpe"] = sharpe_ratio_from_predictions(np.array(y_true), np.array(y_pred))
    return metrics


Device: cpu
Loading data from: feature_selected_dataset.csv
Shape      : (24929, 21)
Target     : target_next_return
# Features : 17
Tickers    : 5


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. LEAKAGE-SAFE SCALING + SEQUENCE BUILDING
#    Features: scaler is fit only on train years for each ticker/fold.
#    Target: y is normalized using train target statistics only.
#    Evaluation: predictions are inverse-transformed before metrics.
# ─────────────────────────────────────────────────────────────────────────────
def build_sequences_for_group(X_vals, y_vals, dates, seq_len):
    """Slide a window over one ticker's data → (n_samples, seq_len, n_features)."""
    Xs, ys, ds = [], [], []
    for i in range(seq_len, len(X_vals)):
        Xs.append(X_vals[i - seq_len : i])
        ys.append(y_vals[i])
        ds.append(dates[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(ds)


def build_scaled_sequences(df, feature_cols, target_col, seq_len, train_years, val_years, test_years):
    X_train_all, y_train_all = [], []
    X_val_all, y_val_all = [], []
    X_test_all, y_test_all = [], []
    meta_train, meta_val, meta_test = [], [], []

    skipped = []

    for ticker, group in df.groupby("ticker"):
        group = group.sort_values("date").reset_index(drop=True)

        tr_mask = group["date"].dt.year.isin(train_years)
        val_mask = group["date"].dt.year.isin(val_years)
        test_mask = group["date"].dt.year.isin(test_years)

        n_train = tr_mask.sum()
        n_val = val_mask.sum()
        n_test = test_mask.sum()

        reason = None
        if n_train < seq_len + 1:
            reason = f"train only {n_train} rows (need {seq_len + 1})"
        elif n_val == 0:
            reason = "no val rows"
        elif n_test == 0:
            reason = "no test rows"

        if reason:
            skipped.append((ticker, reason))
            continue

        scaler = StandardScaler()

        X_full = group[feature_cols].values.astype(np.float32)
        y_full = group[target_col].values.astype(np.float32)
        dates = group["date"].values

        # Leakage-safe: fit only on train rows.
        scaler.fit(X_full[tr_mask])
        X_scaled = scaler.transform(X_full)

        Xs_full, ys_full, ds_full = build_sequences_for_group(X_scaled, y_full, dates, seq_len)
        target_years = pd.to_datetime(ds_full).year

        def _collect(year_list):
            mask = np.isin(target_years, year_list)
            return Xs_full[mask], ys_full[mask], ds_full[mask]

        Xtr, ytr, dtr = _collect(train_years)
        Xva, yva, dva = _collect(val_years)
        Xte, yte, dte = _collect(test_years)

        X_train_all.append(Xtr); y_train_all.append(ytr)
        X_val_all.append(Xva); y_val_all.append(yva)
        X_test_all.append(Xte); y_test_all.append(yte)

        for d in dtr: meta_train.append((ticker, d))
        for d in dva: meta_val.append((ticker, d))
        for d in dte: meta_test.append((ticker, d))

    if skipped:
        print(f"  [skip] {len(skipped)} ticker(s) dropped this fold")

    def _stack(lst):
        valid = [a for a in lst if len(a) > 0]
        return (
            np.concatenate(valid, axis=0)
            if valid else np.empty((0, seq_len, len(feature_cols)), dtype=np.float32)
        )

    def _cat(lst):
        valid = [a for a in lst if len(a) > 0]
        return np.concatenate(valid, axis=0) if valid else np.empty((0,), dtype=np.float32)

    X_train = _stack(X_train_all); y_train_raw = _cat(y_train_all)
    X_val = _stack(X_val_all); y_val_raw = _cat(y_val_all)
    X_test = _stack(X_test_all); y_test_raw = _cat(y_test_all)

    if len(y_train_raw) == 0:
        return None

    # Target normalization uses train target statistics only.
    y_mean = y_train_raw.mean()
    y_std = y_train_raw.std() + 1e-8

    y_train = (y_train_raw - y_mean) / y_std
    y_val = (y_val_raw - y_mean) / y_std
    # Keep y_test raw; model predictions are inverted before metrics.

    meta_train_df = pd.DataFrame(meta_train, columns=["ticker", "date"])
    meta_val_df = pd.DataFrame(meta_val, columns=["ticker", "date"])
    meta_test_df = pd.DataFrame(meta_test, columns=["ticker", "date"])

    return {
        "X_train": X_train,
        "y_train": y_train,
        "y_train_raw": y_train_raw,
        "meta_train": meta_train_df,
        "X_val": X_val,
        "y_val": y_val,
        "y_val_raw": y_val_raw,
        "meta_val": meta_val_df,
        "X_test": X_test,
        "y_test": y_test_raw,
        "meta_test": meta_test_df,
        "y_mean": y_mean,
        "y_std": y_std,
    }


all_years = sorted(df["date"].dt.year.unique())
print(f"Years in dataset: {all_years}")

TRAIN_WINDOW = 5

def make_rolling_folds(all_years, train_window=5):
    """Returns list of (train_years, val_years, test_years) tuples."""
    folds = []
    for i in range(len(all_years) - train_window - 1):
        train_years = list(all_years[i : i + train_window])
        val_years = [all_years[i + train_window]]
        test_years = [all_years[i + train_window + 1]]
        folds.append((train_years, val_years, test_years))
    return folds

rolling_folds = make_rolling_folds(all_years, TRAIN_WINDOW)
print(f"Rolling folds ({len(rolling_folds)} total):")
for k, (tr, va, te) in enumerate(rolling_folds):
    print(f"  Fold {k+1}: train={tr} | val={va} | test={te}")


Years in dataset: [np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Rolling folds (15 total):
  Fold 1: train=[np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009)] | val=[np.int32(2010)] | test=[np.int32(2011)]
  Fold 2: train=[np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010)] | val=[np.int32(2011)] | test=[np.int32(2012)]
  Fold 3: train=[np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011)] | val=[np.int32(2012)] | test=[np.int32(2013)]
  Fold 4: train=[np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012)] | val=[np.int32(2013)] | test=[np.int32(2014)]
  Fold 5: train=[np.int32(2009), np.int32(2

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. TORCH HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).view(-1, 1),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=(device.type == "cuda"))


def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, patience=7, weight_decay=1e-5):
    model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_val_loss = np.inf
    best_state = None
    no_improve = 0

    for epoch in range(epochs):
        model.train()

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(criterion(model(xb), yb).item())

        mean_val = np.mean(val_losses)
        scheduler.step(mean_val)

        if mean_val < best_val_loss:
            best_val_loss = mean_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def predict_model(model, X, batch_size=256):
    model.eval()
    preds = []

    loader = DataLoader(
        torch.tensor(X, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False,
    )

    with torch.no_grad():
        for xb in loader:
            preds.extend(model(xb.to(device)).cpu().numpy().ravel())

    return np.array(preds)


# ─────────────────────────────────────────────────────────────────────────────
# 5. LSTM MODEL ONLY
# ─────────────────────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden=64, num_layers=2, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            n_features,
            hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.norm = nn.LayerNorm(hidden)

        self.head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.norm(out[:, -1, :]))


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. TEST ALL 36 LSTM HYPERPARAMETER COMBINATIONS
# ─────────────────────────────────────────────────────────────────────────────
# 3 seq_len × 3 hidden_dim × 2 lr × 2 dropout = 36 combinations.
# Results are averaged across all rolling folds.

PARAM_GRID = {
    "seq_len": [10, 20, 30],
    "hidden_dim": [32, 64, 128],
    "lr": [1e-3, 5e-4],
    "dropout": [0.2, 0.3],
}

NUM_LAYERS = 2
EPOCHS = 75
BATCH_SIZE = 128
PATIENCE = 10
WEIGHT_DECAY = 1e-5

all_tuning_rows = []

param_combos = list(itertools.product(
    PARAM_GRID["seq_len"],
    PARAM_GRID["hidden_dim"],
    PARAM_GRID["lr"],
    PARAM_GRID["dropout"],
))

print(f"Testing {len(param_combos)} LSTM combinations across {len(rolling_folds)} rolling folds.")

for combo_idx, (seq_len, hidden_dim, lr, dropout) in enumerate(param_combos, start=1):
    print("\n" + "=" * 80)
    print(
        f"COMBO {combo_idx}/{len(param_combos)} | "
        f"seq_len={seq_len}, hidden_dim={hidden_dim}, lr={lr}, dropout={dropout}"
    )
    print("=" * 80)

    for fold_idx, (train_years, val_years, test_years) in enumerate(rolling_folds, start=1):
        print(
            f"  Fold {fold_idx}/{len(rolling_folds)} | "
            f"train={train_years}, val={val_years}, test={test_years}"
        )

        data = build_scaled_sequences(
            df=df,
            feature_cols=feature_cols,
            target_col=TARGET_COL,
            seq_len=seq_len,
            train_years=train_years,
            val_years=val_years,
            test_years=test_years,
        )

        if data is None or len(data["X_train"]) == 0 or len(data["X_val"]) == 0 or len(data["X_test"]) == 0:
            print("    Skipping fold — insufficient data.")
            continue

        X_train = data["X_train"]
        y_train = data["y_train"]
        X_val = data["X_val"]
        y_val = data["y_val"]
        y_val_raw = data["y_val_raw"]
        X_test = data["X_test"]
        y_test = data["y_test"]
        y_mean = data["y_mean"]
        y_std = data["y_std"]

        n_features = X_train.shape[2]

        train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = make_loader(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)

        # Reset seeds so configs are more comparable.
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        model = LSTMModel(
            n_features=n_features,
            hidden=hidden_dim,
            num_layers=NUM_LAYERS,
            dropout=dropout,
        )

        model = train_model(
            model,
            train_loader,
            val_loader,
            epochs=EPOCHS,
            lr=lr,
            patience=PATIENCE,
            weight_decay=WEIGHT_DECAY,
        )

        # Validation predictions are normalized; invert them before metrics.
        y_val_pred_norm = predict_model(model, X_val)
        y_val_pred = y_val_pred_norm * y_std + y_mean
        val_metrics = regression_metrics(
            y_val_raw,
            y_val_pred,
            include_sharpe=(TARGET_COL == "target_next_return"),
        )

        # Test predictions are normalized; invert them before metrics.
        y_test_pred_norm = predict_model(model, X_test)
        y_test_pred = y_test_pred_norm * y_std + y_mean
        test_metrics = regression_metrics(
            y_test,
            y_test_pred,
            include_sharpe=(TARGET_COL == "target_next_return"),
        )

        row = {
            "combo": combo_idx,
            "fold": fold_idx,
            "seq_len": seq_len,
            "hidden_dim": hidden_dim,
            "num_layers": NUM_LAYERS,
            "lr": lr,
            "dropout": dropout,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "patience": PATIENCE,
            "weight_decay": WEIGHT_DECAY,
            "train_years": str(train_years),
            "val_year": str(val_years),
            "test_year": str(test_years),
        }

        for k, v in val_metrics.items():
            row[f"val_{k}"] = v

        for k, v in test_metrics.items():
            row[f"test_{k}"] = v

        all_tuning_rows.append(row)

        print(
            f"    val Sharpe={val_metrics['Sharpe']:.3f}, "
            f"val DA={val_metrics['Directional_Accuracy']:.3f}, "
            f"val RMSE={val_metrics['RMSE']:.5f} | "
            f"test Sharpe={test_metrics['Sharpe']:.3f}, "
            f"test DA={test_metrics['Directional_Accuracy']:.3f}, "
            f"test RMSE={test_metrics['RMSE']:.5f}"
        )

tuning_results = pd.DataFrame(all_tuning_rows)
tuning_results.to_csv("lstm_36_combo_all_fold_results.csv", index=False)

print("\nDone. Full fold-level results saved to lstm_36_combo_all_fold_results.csv")
display(tuning_results.head())


Testing 36 LSTM combinations across 15 rolling folds.

COMBO 1/36 | seq_len=10, hidden_dim=32, lr=0.001, dropout=0.2
  Fold 1/15 | train=[np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009)], val=[np.int32(2010)], test=[np.int32(2011)]
  [skip] 1 ticker(s) dropped this fold
    val Sharpe=1.508, val DA=0.508, val RMSE=0.01566 | test Sharpe=0.070, test DA=0.502, test RMSE=0.01931
  Fold 2/15 | train=[np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010)], val=[np.int32(2011)], test=[np.int32(2012)]
    val Sharpe=0.569, val DA=0.491, val RMSE=0.02241 | test Sharpe=1.023, test DA=0.518, test RMSE=0.01900
  Fold 3/15 | train=[np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011)], val=[np.int32(2012)], test=[np.int32(2013)]
    val Sharpe=0.510, val DA=0.509, val RMSE=0.01900 | test Sharpe=-1.288, test DA=0.475, test RMSE=0.02210
  Fold 4/15 | train=[np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011),

,combo,fold,seq_len,hidden_dim,num_layers,lr,dropout,batch_size,epochs,patience,...,val_MAE,val_R2,val_Directional_Accuracy,val_Sharpe,test_MSE,test_RMSE,test_MAE,test_R2,test_Directional_Accuracy,test_Sharpe
0,1,1,10,32,2,0.001,0.2,128,75,10,...,0.011405,0.002823,0.507937,1.507689,0.000373,0.019314,0.013594,-0.000668,0.501984,0.070161
1,1,2,10,32,2,0.001,0.2,128,75,10,...,0.015488,0.004292,0.491270,0.568772,0.000361,0.019001,0.012054,0.001614,0.518400,1.023404
2,1,3,10,32,2,0.001,0.2,128,75,10,...,0.012079,0.001392,0.508800,0.510095,0.000489,0.022102,0.012316,-0.044673,0.474603,-1.287627
3,1,4,10,32,2,0.001,0.2,128,75,10,...,0.012033,-0.021900,0.476984,-1.018036,0.000289,0.016993,0.011020,-0.002179,0.486508,-0.047023
4,1,5,10,32,2,0.001,0.2,128,75,10,...,0.011003,-0.000772,0.503968,0.026153,0.000301,0.017353,0.012134,0.000734,0.488095,0.528640


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. SUMMARIZE RESULTS + RETURN TOP 3 COMBINATIONS FOR EACH IMPORTANT METRIC
# ─────────────────────────────────────────────────────────────────────────────
group_cols = [
    "combo",
    "seq_len",
    "hidden_dim",
    "num_layers",
    "lr",
    "dropout",
    "batch_size",
    "epochs",
    "patience",
    "weight_decay",
]

metric_cols = [
    "val_MSE", "val_RMSE", "val_MAE", "val_R2", "val_Directional_Accuracy", "val_Sharpe",
    "test_MSE", "test_RMSE", "test_MAE", "test_R2", "test_Directional_Accuracy", "test_Sharpe",
]

combo_summary = (
    tuning_results
    .groupby(group_cols, as_index=False)[metric_cols]
    .mean()
    .sort_values("val_Sharpe", ascending=False)
)

combo_summary.to_csv("lstm_36_combo_summary.csv", index=False)

print("Combination-level summary saved to lstm_36_combo_summary.csv")
display(combo_summary)


def top3(df, metric, higher_is_better=True):
    return df.sort_values(metric, ascending=not higher_is_better).head(3)


important_metrics = {
    "val_Sharpe": True,
    "val_Directional_Accuracy": True,
    "val_RMSE": False,
    "val_MAE": False,
    "val_R2": True,
    "test_Sharpe": True,
    "test_Directional_Accuracy": True,
    "test_RMSE": False,
    "test_MAE": False,
    "test_R2": True,
}

top3_tables = {}

for metric, higher_is_better in important_metrics.items():
    top3_tables[metric] = top3(combo_summary, metric, higher_is_better)

    print("\n" + "=" * 80)
    print(f"TOP 3 BY {metric}")
    print("=" * 80)
    display(top3_tables[metric])


# Most honest model-selection table:
# Use validation metrics to choose hyperparameters, then inspect corresponding test metrics.
selection_cols = [
    "combo",
    "seq_len",
    "hidden_dim",
    "num_layers",
    "lr",
    "dropout",
    "val_Sharpe",
    "val_Directional_Accuracy",
    "val_RMSE",
    "test_Sharpe",
    "test_Directional_Accuracy",
    "test_RMSE",
]

print("\n" + "=" * 80)
print("RECOMMENDED: choose based on validation Sharpe, then inspect test metrics")
print("=" * 80)
display(combo_summary.sort_values("val_Sharpe", ascending=False)[selection_cols].head(10))


Combination-level summary saved to lstm_36_combo_summary.csv


,combo,seq_len,hidden_dim,num_layers,lr,dropout,batch_size,epochs,patience,weight_decay,...,val_MAE,val_R2,val_Directional_Accuracy,val_Sharpe,test_MSE,test_RMSE,test_MAE,test_R2,test_Directional_Accuracy,test_Sharpe
2,3,10,32,2,0.0005,0.2,128,75,10,0.00001,...,0.013652,-0.001043,0.514455,0.730439,0.000464,0.020871,0.013846,-0.006745,0.502894,0.109605
13,14,20,32,2,0.0010,0.3,128,75,10,0.00001,...,0.013654,-0.001196,0.515189,0.722884,0.000464,0.020875,0.013854,-0.006911,0.496011,-0.031380
3,4,10,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013648,-0.000449,0.514670,0.679491,0.000463,0.020865,0.013839,-0.006173,0.503028,0.091956
14,15,20,32,2,0.0005,0.2,128,75,10,0.00001,...,0.013659,-0.001661,0.514111,0.620580,0.000463,0.020861,0.013842,-0.005586,0.501968,0.064630
12,13,20,32,2,0.0010,0.2,128,75,10,0.00001,...,0.013655,-0.001345,0.511469,0.590161,0.000464,0.020882,0.013865,-0.007785,0.494592,-0.099758
15,16,20,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013657,-0.001378,0.511476,0.585737,0.000463,0.020853,0.013833,-0.004878,0.505381,0.187913
27,28,30,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013650,-0.000876,0.510876,0.574018,0.000463,0.020862,0.013834,-0.005736,0.505697,0.154577
26,27,30,32,2,0.0005,0.2,128,75,10,0.00001,...,0.013655,-0.001152,0.511612,0.571421,0.000464,0.020863,0.013838,-0.005723,0.501883,0.146814
5,6,10,64,2,0.0010,0.3,128,75,10,0.00001,...,0.013668,-0.002097,0.506310,0.559392,0.000464,0.020875,0.013862,-0.007091,0.498993,0.042188
28,29,30,64,2,0.0010,0.2,128,75,10,0.00001,...,0.013679,-0.002990,0.507674,0.530425,0.000465,0.020907,0.013900,-0.010660,0.502523,0.111293



TOP 3 BY val_Sharpe


,combo,seq_len,hidden_dim,num_layers,lr,dropout,batch_size,epochs,patience,weight_decay,...,val_MAE,val_R2,val_Directional_Accuracy,val_Sharpe,test_MSE,test_RMSE,test_MAE,test_R2,test_Directional_Accuracy,test_Sharpe
2,3,10,32,2,0.0005,0.2,128,75,10,0.00001,...,0.013652,-0.001043,0.514455,0.730439,0.000464,0.020871,0.013846,-0.006745,0.502894,0.109605
13,14,20,32,2,0.0010,0.3,128,75,10,0.00001,...,0.013654,-0.001196,0.515189,0.722884,0.000464,0.020875,0.013854,-0.006911,0.496011,-0.031380
3,4,10,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013648,-0.000449,0.514670,0.679491,0.000463,0.020865,0.013839,-0.006173,0.503028,0.091956



TOP 3 BY val_Directional_Accuracy


...

,combo,seq_len,hidden_dim,num_layers,lr,dropout,batch_size,epochs,patience,weight_decay,...,val_MAE,val_R2,val_Directional_Accuracy,val_Sharpe,test_MSE,test_RMSE,test_MAE,test_R2,test_Directional_Accuracy,test_Sharpe
25,26,30,32,2,0.0010,0.3,128,75,10,0.00001,...,0.013650,-0.001262,0.507596,0.390479,0.000463,0.020853,0.013833,-0.005047,0.499306,0.086832
15,16,20,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013657,-0.001378,0.511476,0.585737,0.000463,0.020853,0.013833,-0.004878,0.505381,0.187913
27,28,30,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013650,-0.000876,0.510876,0.574018,0.000463,0.020862,0.013834,-0.005736,0.505697,0.154577



TOP 3 BY test_R2


,combo,seq_len,hidden_dim,num_layers,lr,dropout,batch_size,epochs,patience,weight_decay,...,val_MAE,val_R2,val_Directional_Accuracy,val_Sharpe,test_MSE,test_RMSE,test_MAE,test_R2,test_Directional_Accuracy,test_Sharpe
15,16,20,32,2,0.0005,0.3,128,75,10,0.00001,...,0.013657,-0.001378,0.511476,0.585737,0.000463,0.020853,0.013833,-0.004878,0.505381,0.187913
25,26,30,32,2,0.0010,0.3,128,75,10,0.00001,...,0.013650,-0.001262,0.507596,0.390479,0.000463,0.020853,0.013833,-0.005047,0.499306,0.086832
14,15,20,32,2,0.0005,0.2,128,75,10,0.00001,...,0.013659,-0.001661,0.514111,0.620580,0.000463,0.020861,0.013842,-0.005586,0.501968,0.064630



RECOMMENDED: choose based on validation Sharpe, then inspect test metrics


,combo,seq_len,hidden_dim,num_layers,lr,dropout,val_Sharpe,val_Directional_Accuracy,val_RMSE,test_Sharpe,test_Directional_Accuracy,test_RMSE
2,3,10,32,2,0.0005,0.2,0.730439,0.514455,0.020508,0.109605,0.502894,0.020871
13,14,20,32,2,0.0010,0.3,0.722884,0.515189,0.020511,-0.031380,0.496011,0.020875
3,4,10,32,2,0.0005,0.3,0.679491,0.514670,0.020502,0.091956,0.503028,0.020865
14,15,20,32,2,0.0005,0.2,0.620580,0.514111,0.020514,0.064630,0.501968,0.020861
12,13,20,32,2,0.0010,0.2,0.590161,0.511469,0.020513,-0.099758,0.494592,0.020882
15,16,20,32,2,0.0005,0.3,0.585737,0.511476,0.020511,0.187913,0.505381,0.020853
27,28,30,32,2,0.0005,0.3,0.574018,0.510876,0.020506,0.154577,0.505697,0.020862
26,27,30,32,2,0.0005,0.2,0.571421,0.511612,0.020510,0.146814,0.501883,0.020863
5,6,10,64,2,0.0010,0.3,0.559392,0.506310,0.020522,0.042188,0.498993,0.020875
28,29,30,64,2,0.0010,0.2,0.530425,0.507674,0.020527,0.111293,0.502523,0.020907
